# Week 08 Lab — Data Annotation for NLP

## Learning Goals
- Understand why **high-quality annotated data** is the foundation of supervised NLP.
- Apply common annotation schemes: **POS tagging**, **Named Entity Recognition (NER)**, and **sentiment**.
- Measure inter-annotator agreement with **Cohen's κ**, **Fleiss's κ**, and **Krippendorff's α**.
- Identify and analyse sources of **annotator disagreement**.
- Simulate a simple **active learning** loop to reduce annotation cost.
- Reflect on ethical and practical challenges in crowdsourced annotation.

## Part 0 — Setup

In [ ]:
!pip install -q nltk scikit-learn pandas matplotlib krippendorff

In [ ]:
import random
import math
from collections import Counter, defaultdict

import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import krippendorff

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

random.seed(42)
np.random.seed(42)
print("All imports OK.")

---
## Part 1 — Annotation Schemes and Label Sets

Annotation is the process of attaching structured labels to raw text. Different NLP tasks require different label sets and guidelines.

### 1.1 POS Tags (Penn Treebank)

A partial reference for common PTB tags:

| Tag  | Description          | Example       |
|------|----------------------|---------------|
| NN   | Noun, singular       | *dog*         |
| NNS  | Noun, plural         | *dogs*        |
| NNP  | Proper noun, sing.   | *London*      |
| NNPS | Proper noun, plural  | *Vikings*     |
| VB   | Verb, base form      | *run*         |
| VBD  | Verb, past tense     | *ran*         |
| VBG  | Verb, gerund         | *running*     |
| VBZ  | Verb, 3rd sing. pres | *runs*        |
| JJ   | Adjective            | *fast*        |
| RB   | Adverb               | *quickly*     |
| DT   | Determiner           | *the*, *a*    |
| IN   | Preposition/conj.    | *in*, *on*    |
| PRP  | Personal pronoun     | *he*, *she*   |
| CC   | Coordinating conj.   | *and*, *but*  |
| CD   | Cardinal number      | *three*, *10* |

### Task 1A — Manual POS Annotation vs NLTK

**Step 1:** Manually annotate the sentence below using the PTB reference above.  
**Step 2:** Run NLTK's tagger and compare.

Sentence: *"Three scientists from Oxford quickly published a groundbreaking study on climate change."*

In [ ]:
# Your manual annotations — fill in the tags
SENTENCE = "Three scientists from Oxford quickly published a groundbreaking study on climate change ."

MANUAL_TAGS = {
    "Three":          "CD",   # TODO: fill in your tag
    "scientists":     "???",  # TODO
    "from":           "???",  # TODO
    "Oxford":         "???",  # TODO
    "quickly":        "???",  # TODO
    "published":      "???",  # TODO
    "a":              "???",  # TODO
    "groundbreaking": "???",  # TODO
    "study":          "???",  # TODO
    "on":             "???",  # TODO
    "climate":        "???",  # TODO
    "change":         "???",  # TODO
    ".":              ".",
}

# NLTK auto-tagging
tokens = SENTENCE.split()
nltk_tags = dict(nltk.pos_tag(tokens))

# Comparison table
print(f"{'Token':<18} {'Your Tag':<12} {'NLTK Tag':<12} {'Match?'}")
print("-" * 55)
for tok in tokens:
    your = MANUAL_TAGS.get(tok, "?")
    auto = nltk_tags.get(tok, "?")
    match = "✓" if your == auto else "✗"
    print(f"{tok:<18} {your:<12} {auto:<12} {match}")

### Task 1B — BIO-NER Annotation

Apply **BIO** tags with entity types **PER**, **ORG**, **LOC**, **MISC** to the sentences below.

BIO scheme:
- `B-TYPE` — beginning of an entity of the given type
- `I-TYPE` — continuation of the same entity
- `O`       — outside any entity

**Example:**  
`"Barack Obama visited the United Nations ."`  
→ `B-PER I-PER O O B-ORG I-ORG O`

Fill in the table for each sentence:

| # | Sentence | Your BIO annotation |
|---|----------|---------------------|
| 1 | Apple Inc. announced a new partnership with Samsung Electronics in Seoul . | |
| 2 | Dr. Marie Curie was born in Warsaw and later worked at the University of Paris . | |
| 3 | The FIFA World Cup 2026 will be hosted by the United States , Canada , and Mexico . | |
| 4 | Elon Musk 's company SpaceX launched a Falcon 9 rocket from Cape Canaveral . | |
| 5 | The European Central Bank raised interest rates for the fifth time this year . | |

_Fill in your annotations above._

### Task 1C — Sentiment Annotation

Assign a sentiment label (**Positive** / **Neutral** / **Negative**) and confidence (**High** / **Medium** / **Low**) to each review.

| # | Review | Sentiment | Confidence |
|---|--------|-----------|------------|
| 1 | The battery life on this phone is absolutely incredible — it lasted three full days! | | |
| 2 | The package arrived on time and was packaged securely. | | |
| 3 | I expected much better quality for this price. The stitching fell apart after one wash. | | |
| 4 | It works, I guess. Nothing special but nothing broken either. | | |
| 5 | Outstanding customer service — they replaced my defective unit within 24 hours, no questions asked. | | |
| 6 | The manual is confusing and the setup took over two hours. | | |
| 7 | Decent product overall. The colour options could be better. | | |
| 8 | I'm genuinely impressed. Best purchase I've made this year. | | |
| 9 | Returned it immediately. Completely different from what was advertised. | | |
| 10 | Not bad. Does the job. | | |

_Fill in your labels and confidence above._

### Questions
1. What is the difference between **BIO** and **BIOES** tagging schemes for NER? When would BIOES be preferred?
2. Why is it important to write explicit annotation guidelines before labelling starts?
3. Give one example from the sentiment table where the correct label is genuinely ambiguous. Explain why.

_Your answers here._

---
## Part 2 — Inter-Annotator Agreement

**Raw percentage agreement** counts the fraction of items where annotators agree, but ignores how much of that agreement is due to chance. **Cohen's κ** corrects for chance:

$$
\kappa = \frac{p_o - p_e}{1 - p_e}
$$

where $p_o$ is the observed agreement and $p_e$ is the expected chance agreement.

Interpretation guide:
| κ range       | Agreement level |
|---------------|-----------------|
| < 0.00        | Less than chance |
| 0.00 – 0.20   | Slight |
| 0.21 – 0.40   | Fair |
| 0.41 – 0.60   | Moderate |
| 0.61 – 0.80   | Substantial |
| 0.81 – 1.00   | Almost perfect |

In [ ]:
# Coding Task A — Cohen's κ from scratch

def cohens_kappa(ann1: list, ann2: list) -> float:
    """
    Compute Cohen's κ for two annotators with categorical labels.

    Args:
        ann1: list of labels from annotator 1
        ann2: list of labels from annotator 2 (same length)
    Returns:
        κ (float)
    """
    assert len(ann1) == len(ann2), "Annotator lists must have the same length."
    n = len(ann1)
    labels = sorted(set(ann1) | set(ann2))

    # Step 1: Observed agreement p_o
    # TODO: compute the fraction of items where ann1[i] == ann2[i]
    p_o = None  # TODO

    # Step 2: Expected chance agreement p_e
    # p_e = sum over labels k of: P(ann1 == k) * P(ann2 == k)
    p_e = 0.0
    for label in labels:
        p1_k = ann1.count(label) / n  # TODO: fraction of ann1 that assigned label k
        p2_k = ann2.count(label) / n  # TODO: fraction of ann2 that assigned label k
        p_e += p1_k * p2_k

    # Step 3: Compute κ
    # TODO: apply the formula κ = (p_o - p_e) / (1 - p_e)
    kappa = None  # TODO

    return kappa


# --- Test data: two annotators on 20 sentiment items ---
ann_A = ["Pos","Pos","Neg","Neu","Pos","Neg","Neg","Pos","Neu","Pos",
         "Neg","Pos","Pos","Neu","Neg","Pos","Pos","Neg","Neu","Pos"]
ann_B = ["Pos","Neg","Neg","Neu","Pos","Neg","Neu","Pos","Neu","Pos",
         "Neg","Pos","Neu","Neu","Neg","Pos","Pos","Neg","Pos","Pos"]

kappa = cohens_kappa(ann_A, ann_B)
p_o   = sum(a == b for a, b in zip(ann_A, ann_B)) / len(ann_A)
print(f"Observed agreement (p_o) : {p_o:.3f}")
print(f"Cohen's κ                : {kappa:.3f}" if kappa is not None else "Cohen's κ: NOT IMPLEMENTED")

In [ ]:
# Coding Task B — Fleiss's κ for N annotators

def fleiss_kappa(ratings: list, n_annotators: int) -> float:
    """
    Compute Fleiss's κ for N annotators rating M items with categorical labels.

    Args:
        ratings : list of M lists, each inner list has n_annotators label strings
        n_annotators : number of annotators per item
    Returns:
        κ (float)
    """
    # Collect all labels
    all_labels = sorted({lbl for item in ratings for lbl in item})
    M = len(ratings)
    n = n_annotators

    # Step 1: Build count matrix  shape (M, |labels|)
    # count_matrix[i][j] = number of annotators who assigned label j to item i
    count_matrix = []
    for item in ratings:
        row = {lbl: 0 for lbl in all_labels}
        for lbl in item:
            row[lbl] += 1
        count_matrix.append([row[lbl] for lbl in all_labels])
    count_matrix = np.array(count_matrix, dtype=float)

    # Step 2: P_i = agreement for item i = (sum_j n_ij*(n_ij-1)) / (n*(n-1))
    # TODO: compute P_bar = mean over all items of P_i
    P_i = np.array([
        (np.sum(row ** 2) - n) / (n * (n - 1))
        for row in count_matrix
    ])
    P_bar = None  # TODO: np.mean(P_i)

    # Step 3: p_j = marginal proportion of label j
    # TODO: compute P_e = sum_j p_j^2
    p_j = count_matrix.sum(axis=0) / (M * n)
    P_e = None  # TODO: np.sum(p_j ** 2)

    # Step 4: κ = (P_bar - P_e) / (1 - P_e)
    kappa = None  # TODO

    return kappa


# --- Test data: 3 annotators on 20 items ---
ratings_3ann = [list(x) for x in zip(ann_A, ann_B,
    ["Pos","Pos","Neg","Neu","Pos","Neg","Neu","Pos","Neu","Pos",
     "Neg","Pos","Pos","Neg","Neg","Pos","Neu","Neg","Neu","Pos"]
)]

fk = fleiss_kappa(ratings_3ann, n_annotators=3)
print(f"Fleiss's κ (3 annotators): {fk:.3f}" if fk is not None else "Fleiss's κ: NOT IMPLEMENTED")

In [ ]:
# Coding Task C — Krippendorff's α for ordinal data

# Severity ratings (1–5 scale) from 3 annotators on 15 items
# NaN indicates that annotator did not rate that item
severity_ratings = np.array([
    [5, 4, 5],
    [3, 3, 4],
    [1, 2, 1],
    [4, 5, 4],
    [2, 2, 3],
    [5, 5, 5],
    [1, 1, 2],
    [3, 4, 3],
    [4, 4, 4],
    [2, 3, 2],
    [5, 4, np.nan],  # annotator 3 skipped this item
    [1, 1, 1],
    [3, 3, 3],
    [4, 3, 4],
    [2, 2, 2],
], dtype=float)

# krippendorff library expects shape (n_annotators, n_items)
# Annotators are rows, items are columns
reliability_data = severity_ratings.T  # shape: (3, 15)

# TODO: call krippendorff.alpha() with level_of_measurement='ordinal'
# See: https://github.com/pln-fing-udelar/fast-krippendorff
alpha = None  # TODO

print(f"Krippendorff's α (ordinal): {alpha:.3f}" if alpha is not None else "α: NOT IMPLEMENTED")

In [ ]:
# Coding Task D — Confusion analysis between two annotators

def annotator_confusion(ann1: list, ann2: list, labels: list) -> pd.DataFrame:
    """
    Build a confusion matrix (ann1 as rows, ann2 as columns).
    Returns a DataFrame with raw counts.
    """
    matrix = pd.DataFrame(0, index=labels, columns=labels)
    # TODO: for each pair (ann1[i], ann2[i]) increment matrix[ann1[i]][ann2[i]]
    for a, b in zip(ann1, ann2):
        pass  # TODO

    return matrix

labels = ["Pos", "Neu", "Neg"]
confusion = annotator_confusion(ann_A, ann_B, labels)
print("Annotator confusion matrix (rows = Ann A, cols = Ann B):")
print(confusion)

# TODO: identify the most-confused label pair
# (the off-diagonal cell with the highest count)
print("\nMost confused pair: ???")

### IAA Summary Table

Fill in after completing Tasks 2A–2C:

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Raw agreement (A vs B) | | |
| Cohen's κ (A vs B) | | |
| Fleiss's κ (3 annotators) | | |
| Krippendorff's α (ordinal) | | |

### Questions
1. A pair of annotators achieves 85% raw agreement on a binary task where one label covers 90% of items. Compute the expected chance agreement $p_e$ and the resulting κ by hand. What does this tell you?
2. What κ value is generally considered "substantial" agreement? What threshold is often required before publishing a dataset?
3. When would Krippendorff's α be preferred over Cohen's κ?

_Your answers here._

---
## Part 3 — Annotating a NER Dataset

We will apply BIO-NER annotation to a small news corpus and compute IAA between a human annotator (you, represented by `GOLD_TAGS` below) and a simple rule-based baseline annotator.

In [ ]:
# NER corpus — pre-tokenised sentences and gold BIO tags

NER_SENTENCES = [
    ["Apple", "Inc.", "announced", "a", "new", "partnership", "with",
     "Samsung", "Electronics", "in", "Seoul", "."],
    ["Dr.", "Marie", "Curie", "was", "born", "in", "Warsaw", "and",
     "later", "worked", "at", "the", "University", "of", "Paris", "."],
    ["The", "FIFA", "World", "Cup", "2026", "will", "be", "hosted", "by",
     "the", "United", "States", ",", "Canada", ",", "and", "Mexico", "."],
    ["Elon", "Musk", "'s", "company", "SpaceX", "launched", "a",
     "Falcon", "9", "rocket", "from", "Cape", "Canaveral", "."],
    ["The", "European", "Central", "Bank", "raised", "interest", "rates",
     "for", "the", "fifth", "time", "this", "year", "."],
]

# Gold annotations — fill in your BIO tags from Task 1B
# Use: B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, B-MISC, I-MISC, O
GOLD_TAGS = [
    ["B-ORG", "I-ORG", "O", "O", "O", "O", "O",
     "B-ORG", "I-ORG", "O", "B-LOC", "O"],
    ["O", "B-PER", "I-PER", "O", "O", "O", "B-LOC", "O",
     "O", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "O"],
    # TODO: fill in sentences 3-5 from your Task 1B annotations
    ["O", "B-MISC", "I-MISC", "I-MISC", "I-MISC", "O", "O", "O", "O",
     "O", "B-LOC", "I-LOC", "O", "B-LOC", "O", "O", "B-LOC", "O"],
    ["B-PER", "I-PER", "O", "O", "B-ORG", "O", "O",
     "B-MISC", "I-MISC", "O", "O", "B-LOC", "I-LOC", "O"],
    ["O", "B-ORG", "I-ORG", "I-ORG", "O", "O", "O",
     "O", "O", "O", "O", "O", "O", "O"],
]

# Quick sanity check
for i, (sent, tags) in enumerate(zip(NER_SENTENCES, GOLD_TAGS)):
    assert len(sent) == len(tags), f"Sentence {i+1}: token/tag length mismatch"

print("Gold annotations loaded. Token counts:", [len(s) for s in NER_SENTENCES])

In [ ]:
# Coding Task A — Rule-based "second annotator" (capitalisation heuristic)

def capitalisation_tagger(sentences: list) -> list:
    """
    Naive NER tagger: assigns B-MISC to any capitalised non-first token,
    I-MISC if the previous token was also capitalised, O otherwise.
    First token of each sentence is always O (to avoid sentence-initial bias).

    Returns a list of lists of BIO tags (same structure as GOLD_TAGS).
    """
    all_tags = []
    for sent in sentences:
        tags = []
        for i, tok in enumerate(sent):
            if i == 0 or tok in {".", ",", "'s", "the", "The"}:
                tags.append("O")
            elif tok[0].isupper():
                # TODO: assign B-MISC or I-MISC depending on the previous tag
                prev_tag = tags[-1] if tags else "O"
                if prev_tag in {"B-MISC", "I-MISC"}:
                    tags.append("I-MISC")  # continuation
                else:
                    tags.append("B-MISC")  # new entity
            else:
                tags.append("O")
        all_tags.append(tags)
    return all_tags

AUTO_TAGS = capitalisation_tagger(NER_SENTENCES)

# Display side-by-side
for i, (sent, gold, auto) in enumerate(zip(NER_SENTENCES, GOLD_TAGS, AUTO_TAGS)):
    print(f"\n--- Sentence {i+1} ---")
    print(f"{'Token':<20} {'Gold':<12} {'Auto':<12} {'Match?'}")
    print("-" * 50)
    for tok, g, a in zip(sent, gold, auto):
        match = "✓" if g == a else "✗"
        print(f"{tok:<20} {g:<12} {a:<12} {match}")

In [ ]:
# Coding Task B — Token-level IAA (Cohen's κ)

# Flatten all tags into two parallel lists
flat_gold = [tag for sent_tags in GOLD_TAGS for tag in sent_tags]
flat_auto = [tag for sent_tags in AUTO_TAGS for tag in sent_tags]

# TODO: call your cohens_kappa() from Part 2
token_kappa = None  # TODO: cohens_kappa(flat_gold, flat_auto)
token_agree = sum(g == a for g, a in zip(flat_gold, flat_auto)) / len(flat_gold)

print(f"Total tokens             : {len(flat_gold)}")
print(f"Token-level raw agreement: {token_agree:.3f}")
print(f"Token-level Cohen's κ   : {token_kappa:.3f}" if token_kappa is not None else "κ: NOT IMPLEMENTED")

In [ ]:
# Coding Task C — Span-level Precision, Recall, F1

def bio_to_spans(tags: list, tokens: list) -> set:
    """
    Convert BIO tags to a set of (start, end, entity_type) tuples.
    start/end are token indices (end is exclusive).
    E.g. B-PER at 1, I-PER at 2 → (1, 3, 'PER')
    """
    spans = set()
    start, etype = None, None

    for i, tag in enumerate(tags):
        if tag.startswith("B-"):
            if start is not None:
                spans.add((start, i, etype))
            start = i
            etype = tag[2:]
        elif tag.startswith("I-") and start is not None and tag[2:] == etype:
            pass  # continuing the same entity
        else:
            if start is not None:
                spans.add((start, i, etype))
                start, etype = None, None

    if start is not None:
        spans.add((start, len(tags), etype))

    return spans


def span_f1(gold_spans: set, pred_spans: set) -> tuple:
    """
    Compute span-level precision, recall, F1.
    Returns (precision, recall, f1).
    """
    tp = len(gold_spans & pred_spans)
    # TODO: precision = tp / len(pred_spans)  (if pred_spans non-empty)
    # TODO: recall    = tp / len(gold_spans)   (if gold_spans non-empty)
    # TODO: f1        = 2 * precision * recall / (precision + recall)
    precision = None  # TODO
    recall    = None  # TODO
    f1        = None  # TODO
    return precision, recall, f1


# Flatten tokens and compute spans across the full corpus
all_gold_spans, all_pred_spans = set(), set()
offset = 0
for sent, gold, auto in zip(NER_SENTENCES, GOLD_TAGS, AUTO_TAGS):
    # Shift span indices by the cumulative offset
    g_spans = {(s + offset, e + offset, t) for s, e, t in bio_to_spans(gold, sent)}
    p_spans = {(s + offset, e + offset, t) for s, e, t in bio_to_spans(auto, sent)}
    all_gold_spans |= g_spans
    all_pred_spans |= p_spans
    offset += len(sent)

p, r, f = span_f1(all_gold_spans, all_pred_spans)

print(f"Gold spans : {len(all_gold_spans)}")
print(f"Auto spans : {len(all_pred_spans)}")
if p is not None:
    print(f"\nSpan-level Precision : {p:.3f}")
    print(f"Span-level Recall    : {r:.3f}")
    print(f"Span-level F1        : {f:.3f}")
else:
    print("span_f1 not implemented yet.")

### Questions
1. Why is token-level κ often misleadingly high for NER even when annotators disagree on entity boundaries?
2. What information is lost when you compute token-level agreement instead of span-level agreement?
3. The capitalisation tagger assigns only `MISC` labels. How does this limitation affect precision and recall for `PER`, `ORG`, `LOC` entity types?

_Your answers here._

---
## Part 4 — Sentiment Annotation and Adjudication

We simulate a three-annotator sentiment annotation task with varying noise levels and practice adjudication.

In [ ]:
# Coding Task A — Simulate three annotators with different noise levels

SENT_LABELS = ["Pos", "Neu", "Neg"]

# "True" labels for 20 items
TRUE_LABELS = ["Pos","Pos","Neg","Neu","Pos","Neg","Neg","Pos","Neu","Pos",
               "Neg","Pos","Pos","Neu","Neg","Pos","Neu","Neg","Neu","Pos"]


def noisy_annotator(true_labels: list, noise: float, seed: int) -> list:
    """
    Simulate an annotator who assigns the true label with probability (1 - noise)
    and a random *different* label with probability noise.
    """
    rng = random.Random(seed)
    result = []
    for lbl in true_labels:
        if rng.random() < noise:
            # Pick a random label that is NOT the true one
            wrong = [l for l in SENT_LABELS if l != lbl]
            result.append(rng.choice(wrong))
        else:
            result.append(lbl)
    return result


# Annotator noise levels
ann1 = noisy_annotator(TRUE_LABELS, noise=0.05, seed=1)  # careful annotator
ann2 = noisy_annotator(TRUE_LABELS, noise=0.20, seed=2)  # moderate noise
ann3 = noisy_annotator(TRUE_LABELS, noise=0.35, seed=3)  # noisy annotator

print("Item  True   Ann1   Ann2   Ann3")
print("-" * 35)
for i, (t, a1, a2, a3) in enumerate(zip(TRUE_LABELS, ann1, ann2, ann3), 1):
    marker = " *" if not (a1 == a2 == a3) else ""
    print(f"{i:>4}  {t:<6} {a1:<6} {a2:<6} {a3:<6}{marker}")

In [ ]:
# Coding Task B — Majority vote aggregation

def majority_vote(annotations: list) -> list:
    """
    Given a list of N annotator label lists (each of length M items),
    return a list of M aggregated labels using majority vote.
    Ties are broken by the order in SENT_LABELS.
    """
    n_items = len(annotations[0])
    result = []
    for i in range(n_items):
        item_labels = [annotations[j][i] for j in range(len(annotations))]
        # TODO: find the most common label; break ties using SENT_LABELS order
        counts = Counter(item_labels)
        voted = None  # TODO: max by count, then by SENT_LABELS index for ties
        result.append(voted)
    return result


all_annotations = [ann1, ann2, ann3]
mv_labels = majority_vote(all_annotations)

# Compare with true labels
mv_accuracy = sum(mv == t for mv, t in zip(mv_labels, TRUE_LABELS)) / len(TRUE_LABELS)
print(f"Majority-vote accuracy vs. true labels: {mv_accuracy:.3f}")

print("\nItem  MV     True   Match?")
print("-" * 30)
for i, (mv, t) in enumerate(zip(mv_labels, TRUE_LABELS), 1):
    match = "✓" if mv == t else "✗"
    print(f"{i:>4}  {mv:<6} {t:<6} {match}")

In [ ]:
# Coding Task C — Fleiss's κ for the three-annotator task

ratings_part4 = [list(triple) for triple in zip(ann1, ann2, ann3)]

# TODO: call your fleiss_kappa() from Part 2
fk_part4 = None  # TODO: fleiss_kappa(ratings_part4, n_annotators=3)
print(f"Fleiss's κ (3 annotators, 20 items): {fk_part4:.3f}" if fk_part4 is not None else "κ: NOT IMPLEMENTED")

# Also compute pairwise Cohen's κ
pairs = [("Ann1 vs Ann2", ann1, ann2),
         ("Ann1 vs Ann3", ann1, ann3),
         ("Ann2 vs Ann3", ann2, ann3)]

print("\nPairwise Cohen's κ:")
for name, a, b in pairs:
    k = cohens_kappa(a, b)
    print(f"  {name}: {k:.3f}" if k is not None else f"  {name}: NOT IMPLEMENTED")

In [ ]:
# Coding Task D — Adjudication: find full-disagreement items

def find_full_disagreements(annotations: list) -> list:
    """
    Return the indices (0-based) of items where ALL annotators assigned
    different labels (no two annotators agree).
    """
    n_items = len(annotations[0])
    disagreements = []
    for i in range(n_items):
        item_labels = [annotations[j][i] for j in range(len(annotations))]
        # TODO: check if all labels are different (i.e. no duplicates)
        if len(set(item_labels)) == len(item_labels):  # all distinct
            disagreements.append(i)
    return disagreements


full_disagree = find_full_disagreements(all_annotations)
print(f"Items with full 3-way disagreement: {[i+1 for i in full_disagree]}")

for idx in full_disagree:
    print(f"\n  Item {idx+1}: Ann1={ann1[idx]}, Ann2={ann2[idx]}, Ann3={ann3[idx]}, True={TRUE_LABELS[idx]}")
    print(f"  → Proposed adjudication rule: ???  (fill in your suggestion)")

### Questions
1. How does majority voting differ from Dawid–Skene expectation-maximisation for aggregating crowd labels?
2. What are two strategies for handling items where all annotators disagree after majority voting?
3. How would you design an annotation study to minimise annotator fatigue?

_Your answers here._

---
## Part 5 — Active Learning for Annotation

**Active learning** reduces annotation cost by selecting the most informative examples to label next. Instead of labelling examples at random, the learner queries examples it is most uncertain about.

**Least-confidence** uncertainty sampling selects examples where the model's top predicted class probability is lowest:

$$
x^* = \arg\min_{x} \; P(\hat{y} \mid x)
$$

In [ ]:
# Dataset for active learning — a small sentiment corpus

# 200 short synthetic reviews with binary sentiment labels (1=Pos, 0=Neg)
_pos_templates = [
    "I love this product", "Absolutely amazing quality", "Best purchase ever",
    "Highly recommend it", "Works perfectly fine", "Great value for money",
    "Exceeded my expectations", "Very impressed overall", "Fantastic experience",
    "Will buy again definitely",
]
_neg_templates = [
    "Terrible quality product", "Complete waste of money", "Broke after one day",
    "Would not recommend it", "Very disappointed overall", "Does not work at all",
    "Poor customer service experience", "Nothing like the description", 
    "Returned it immediately", "Absolute rubbish honestly",
]

rng = random.Random(0)
ALL_TEXTS, ALL_LABELS = [], []
for _ in range(100):
    t = rng.choice(_pos_templates)
    filler = rng.choice(["", " really", " truly", " honestly", " definitely"])
    ALL_TEXTS.append(t + filler)
    ALL_LABELS.append(1)
for _ in range(100):
    t = rng.choice(_neg_templates)
    filler = rng.choice(["", " really", " totally", " absolutely", " completely"])
    ALL_TEXTS.append(t + filler)
    ALL_LABELS.append(0)

combined = list(zip(ALL_TEXTS, ALL_LABELS))
rng.shuffle(combined)
ALL_TEXTS, ALL_LABELS = zip(*combined)
ALL_TEXTS, ALL_LABELS = list(ALL_TEXTS), list(ALL_LABELS)

print(f"Dataset: {len(ALL_TEXTS)} examples, {sum(ALL_LABELS)} positive, {len(ALL_LABELS)-sum(ALL_LABELS)} negative")
print("Sample:", ALL_TEXTS[0], "→", ALL_LABELS[0])

In [ ]:
# Coding Task A — Train a baseline classifier on a seed set

# Split: first 20 = seed (labelled), rest = unlabelled pool, last 40 = test
SEED_SIZE  = 20
TEST_SIZE  = 40
seed_texts  = ALL_TEXTS[:SEED_SIZE]
seed_labels = ALL_LABELS[:SEED_SIZE]
pool_texts  = ALL_TEXTS[SEED_SIZE:-TEST_SIZE]
pool_labels = ALL_LABELS[SEED_SIZE:-TEST_SIZE]   # used only for oracle labelling
test_texts  = ALL_TEXTS[-TEST_SIZE:]
test_labels = ALL_LABELS[-TEST_SIZE:]

# Shared TF-IDF vectoriser (fit once on all texts to fix the vocabulary)
vectoriser = TfidfVectorizer(max_features=500)
vectoriser.fit(ALL_TEXTS)

def train_classifier(texts: list, labels: list):
    """Train a logistic regression classifier and return it."""
    X = vectoriser.transform(texts)
    clf = LogisticRegression(max_iter=500, random_state=42)
    clf.fit(X, labels)
    return clf


def evaluate_classifier(clf, texts: list, labels: list) -> float:
    """Return accuracy on the given set."""
    X = vectoriser.transform(texts)
    preds = clf.predict(X)
    return accuracy_score(labels, preds)


# Baseline: train on seed only
baseline_clf = train_classifier(seed_texts, seed_labels)
baseline_acc = evaluate_classifier(baseline_clf, test_texts, test_labels)
print(f"Baseline accuracy (seed={SEED_SIZE}): {baseline_acc:.3f}")

In [ ]:
# Coding Task B — Least-confidence uncertainty sampling

def least_confidence_indices(clf, texts: list, batch_size: int) -> list:
    """
    Return the indices of the `batch_size` examples from `texts` where
    the classifier is least confident (lowest max class probability).

    Args:
        clf        : trained sklearn classifier with predict_proba
        texts      : list of unlabelled text strings
        batch_size : number of examples to select
    Returns:
        list of integer indices into `texts`
    """
    X = vectoriser.transform(texts)
    proba = clf.predict_proba(X)  # shape (n, n_classes)

    # TODO: compute max probability per example (the confidence score)
    max_proba = None  # TODO: np.max(proba, axis=1)

    # TODO: return indices of the batch_size examples with the LOWEST max_proba
    indices = None  # TODO: np.argsort(max_proba)[:batch_size].tolist()
    return indices


# Quick test
test_indices = least_confidence_indices(baseline_clf, pool_texts, batch_size=5)
print("5 most uncertain pool examples:")
for idx in (test_indices or []):
    X_i = vectoriser.transform([pool_texts[idx]])
    conf = baseline_clf.predict_proba(X_i)[0].max()
    print(f"  [{idx}] conf={conf:.3f}  '{pool_texts[idx]}'")

In [ ]:
# Coding Task C — Active learning loop (5 rounds, batch size 10)

ROUNDS     = 5
BATCH_SIZE = 10

def active_learning_loop(strategy: str) -> tuple:
    """
    Run active learning for ROUNDS rounds with the given query strategy.

    strategy: 'uncertainty' or 'random'

    Returns:
        (labelled_counts, accuracies) — lists of length ROUNDS+1
    """
    # Start with a copy of the seed set
    labelled_texts  = list(seed_texts)
    labelled_labels = list(seed_labels)
    pool_t = list(pool_texts)
    pool_l = list(pool_labels)

    labelled_counts = [len(labelled_texts)]
    accuracies = []

    # Round 0: evaluate on seed only
    clf = train_classifier(labelled_texts, labelled_labels)
    accuracies.append(evaluate_classifier(clf, test_texts, test_labels))

    for _ in range(ROUNDS):
        if not pool_t:
            break

        if strategy == 'uncertainty':
            # TODO: use least_confidence_indices to select BATCH_SIZE examples
            selected_indices = None  # TODO
        else:  # random
            # TODO: randomly select BATCH_SIZE indices from range(len(pool_t))
            selected_indices = None  # TODO: random.sample(range(len(pool_t)), min(BATCH_SIZE, len(pool_t)))

        if selected_indices is None:
            break

        # Move selected examples from pool to labelled set
        selected_indices_sorted = sorted(selected_indices, reverse=True)
        for idx in selected_indices_sorted:
            labelled_texts.append(pool_t.pop(idx))
            labelled_labels.append(pool_l.pop(idx))

        # Retrain and evaluate
        clf = train_classifier(labelled_texts, labelled_labels)
        acc = evaluate_classifier(clf, test_texts, test_labels)
        accuracies.append(acc)
        labelled_counts.append(len(labelled_texts))

    return labelled_counts, accuracies


counts_al, accs_al     = active_learning_loop('uncertainty')
counts_rand, accs_rand = active_learning_loop('random')

print(f"{'Round':<8} {'AL labelled':>12} {'AL acc':>8} {'Rand acc':>10}")
print("-" * 45)
for i, (na, aa, nr, ar) in enumerate(zip(counts_al, accs_al, counts_rand, accs_rand)):
    print(f"{i:<8} {na:>12} {aa:>8.3f} {ar:>10.3f}")

In [ ]:
# Plot learning curves: uncertainty sampling vs random sampling

fig, ax = plt.subplots(figsize=(8, 4))

# TODO: plot accs_al vs counts_al  (label='Uncertainty sampling')
# TODO: plot accs_rand vs counts_rand  (label='Random sampling')
# Add axis labels, title, legend, and grid
ax.set_xlabel("Number of labelled examples")
ax.set_ylabel("Test accuracy")
ax.set_title("Active Learning: Uncertainty Sampling vs Random Sampling")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("active_learning_curves.png", dpi=100)
plt.show()

### Questions
1. What is the **cold-start problem** in active learning and how is it typically addressed?
2. Name two strategies other than least-confidence uncertainty sampling used in active learning.
3. What is a potential bias introduced by active learning that could hurt model generalisation?

_Your answers here._

---
## Part 6 — Crowdsourced Annotation: Ethics and Quality

### Discussion Tasks

Answer each question in your own words (3–5 sentences each).

**6.1 Annotator Biases**  
Identify **three potential biases** that might arise when crowdsourcing sentiment annotation on a platform like Amazon Mechanical Turk. For each bias, describe how it could distort the resulting dataset.

_Your answer here._

---

**6.2 Gold Standard Items (Honeypots)**  
Explain the concept of **gold standard items** (honeypots) in crowdsourced annotation. How are they used to detect and filter low-quality annotators? What are the risks of relying on them too heavily?

_Your answer here._

---

**6.3 Annotator Compensation**  
Describe one **fairness concern** related to paying crowdworkers minimum wage or below for complex linguistic annotation tasks. How might this affect data quality and what alternatives exist?

_Your answer here._

---

**6.4 Annotation Schema and Downstream Tasks**  
How does **annotation schema design** (choice of label set, granularity, guidelines) affect the usefulness of the resulting dataset for downstream NLP tasks? Give a concrete example where a coarse vs. fine-grained schema would produce different outcomes.

_Your answer here._

---
## Submission

- Complete all `TODO` sections in the code cells.
- Answer all written questions in the markdown cells above.
- Include in your submission:
  - The **IAA summary table** from Part 2
  - The **NER span-level F1 table** from Part 3
  - The **active learning learning-curve plot** from Part 5 (`active_learning_curves.png`)
- Submit your completed `.ipynb` file as instructed.